# HousePrice AI — Phase 2 & 3: Data Preprocessing & Feature Engineering

## 1. Overview
This notebook implements the complete data preprocessing and feature engineering pipeline:
1. Selection of baseline features.
2. Handling missing target and input values.
3. Non-finite value validation.
4. Feature engineering (`HouseAge`, `AreaPerBedroom`, `TotalBathrooms`, `TotalSF`).
5. Clean splitting into 70% Train / 15% Validation / 15% Test without data leakage.
6. Manual Feature Scaling via `StandardScaler`.

---

In [15]:
import os
import sys
import numpy as np
import pandas as pd

sys.path.append(os.path.abspath('..'))

from src.data_loader import load_dataset
from src.preprocessing import StandardScaler, select_features, handle_missing_values, remove_missing_target, validate_numerical
from src.feature_engineering import add_house_age, add_area_per_bedroom, add_total_bathrooms, add_total_sf, create_feature_set
from src.data_split import train_val_test_split

## 2. Loading the Dataset

In [16]:
data_path = '../data/raw/train.csv'
if not os.path.exists(data_path):
    alt_path = '../data/House Price Prediction Dataset.csv'
    df_raw = load_dataset(alt_path)
else:
    df_raw = load_dataset(data_path)

print(f'Raw data shape: {df_raw.shape}')

[INFO] Dataset loaded successfully from: ../data/raw/train.csv
   Shape: 1460 rows x 81 columns
Raw data shape: (1460, 81)


## 3. Handling Missing Target and Feature Values

Rows with missing target values must be dropped because we cannot supervise training without true prices.
Missing features are imputed using the column median.

In [17]:
target_col = 'SalePrice' if 'SalePrice' in df_raw.columns else ('Price' if 'Price' in df_raw.columns else df_raw.columns[-1])
df_clean = remove_missing_target(df_raw, target_col)

# Apply feature engineering
if 'YearBuilt' in df_clean.columns:
    df_clean = add_house_age(df_clean, reference_year=2010)
if 'GrLivArea' in df_clean.columns and 'BedroomAbvGr' in df_clean.columns:
    df_clean = add_area_per_bedroom(df_clean)
if 'FullBath' in df_clean.columns:
    df_clean = add_total_bathrooms(df_clean)
df_clean = add_total_sf(df_clean)

[INFO] No missing values in target column 'SalePrice'.
[INFO] Added 'HouseAge' (reference year=2010)
   Range: 0 - 138 years
   [INFO] 6 homes have 0 bedrooms; AreaPerBedroom set to GrLivArea for those.
[INFO] Added 'AreaPerBedroom'
   Range: 236.8 - 2234.0 sq ft/bedroom
[INFO] Added 'TotalBathrooms' (FullBath + 0.5 * HalfBath)
   Range: 0.0 - 3.5
[INFO] Added 'TotalSF' = TotalBsmtSF + 1stFlrSF + 2ndFlrSF
   Range: 334 - 11752 sq ft


## 4. Selecting Feature Matrix $X$ and Target $y$

In [18]:
baseline_features = ['GrLivArea', 'OverallQual', 'YearBuilt', 'BedroomAbvGr', 'FullBath', 'GarageCars']
available_features = [f for f in baseline_features if f in df_clean.columns]

if not available_features:
    # Fallback to numeric columns if alternative dataset loaded
    available_features = [c for c in df_clean.select_dtypes(include=[np.number]).columns if c not in [target_col, 'Id']][:6]

print(f'Selected features: {available_features}')
X_df = df_clean[available_features].copy()
X_df, fill_values = handle_missing_values(X_df, strategy='median')

X = X_df.values.astype(np.float64)
y = df_clean[target_col].values.astype(np.float64)

assert validate_numerical(X), 'X contains non-finite values!'
assert validate_numerical(y), 'y contains non-finite values!'

Selected features: ['GrLivArea', 'OverallQual', 'YearBuilt', 'BedroomAbvGr', 'FullBath', 'GarageCars']
[INFO] No missing values found in numeric columns.
[INFO] Array is clean: no NaN or inf values.
[INFO] Array is clean: no NaN or inf values.


## 5. Train / Validation / Test Splitting

We split the dataset into:
- **70% Training:** Used for parameter estimation (weights and bias) and scaling statistics.
- **15% Validation:** Used for model selection and hyperparameter tuning.
- **15% Test:** Held-out strictly for the final unbiased evaluation.

In [19]:
X_train, X_val, X_test, y_train, y_val, y_test = train_val_test_split(
    X, y, train_ratio=0.70, val_ratio=0.15, test_ratio=0.15, random_seed=42
)

DATA SPLIT SUMMARY
  Total samples : 1460
  Training      :  1021  (69.9%)
  Validation    :   219  (15.0%)
  Test          :   220  (15.1%)
  Random seed   : 42
  No overlap    : verified


## 6. Feature Scaling from Scratch (`StandardScaler`)

Standardization scales each feature to zero mean and unit variance:
$$z = \frac{x - \mu}{\sigma}$$

> **Important Principle:** To prevent data leakage, $\mu$ and $\sigma$ are computed strictly from the training set (`X_train`), and reused to scale `X_val` and `X_test`.

In [20]:
scaler_X = StandardScaler()
X_train_scaled = scaler_X.fit_transform(X_train)
X_val_scaled = scaler_X.transform(X_val)
X_test_scaled = scaler_X.transform(X_test)

print('Feature means (train):', np.round(scaler_X.mean, 3))
print('Feature stds  (train):', np.round(scaler_X.std, 3))
print('Scaled train mean (should be ~0):', np.round(np.mean(X_train_scaled, axis=0), 3))
print('Scaled train std  (should be ~1):', np.round(np.std(X_train_scaled, axis=0), 3))

Feature means (train): [1.512690e+03 6.090000e+00 1.971452e+03 2.850000e+00 1.553000e+00
 1.754000e+00]
Feature stds  (train): [529.882   1.422  29.839   0.799   0.55    0.764]
Scaled train mean (should be ~0): [ 0.  0.  0. -0. -0. -0.]
Scaled train std  (should be ~1): [1. 1. 1. 1. 1. 1.]


## 7. Saving Processed Splits
We save the processed arrays for reproducible experiments across notebooks.

In [21]:
os.makedirs('../data/processed', exist_ok=True)
np.savez('../data/processed/splits_baseline.npz',
         X_train=X_train_scaled, X_val=X_val_scaled, X_test=X_test_scaled,
         y_train=y_train, y_val=y_val, y_test=y_test,
         features=available_features)
print('[SUCCESS] Processed dataset saved to data/processed/splits_baseline.npz')

[SUCCESS] Processed dataset saved to data/processed/splits_baseline.npz
